# 02 — ObjectScale S3 Throttler
**Objective:** validate native S3 throttling — a noisy tenant is capped while a critical workload stays inside SLA and recovers.

**Customer sees:** four measured phases with verdicts against approved thresholds, and a live TPS/latency chart.
**TC:** TC2.1–TC2.9

**Phases:** 1 baseline (critical GET only) → 2 contention (+noisy PUTs, dry-run rules) → 3 enforced (429 limit on noisy PUT) → 4 recovery (noisy stopped, critical continues).

> Caveat: on this lab build rules propagate with a cache delay and enforcement may need a datahead reload — verdicts report INCONCLUSIVE rather than fabricating throttling (see reports/environment.md).

## Step 1 — Session, tenant isolation, effective policy
Prove noisy/critical are distinct IAM principals, then **read** the current throttler config — rules are never changed outside this approved lab workflow.

In [ ]:
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz, s3util
import poc.spark_helpers as sh
spark = sh.build_session("obs-poc")
viz.table(__import__("pandas").DataFrame(sh.versions(spark).items(), columns=["component","version"]))
print("users:", config.masked_users())
print("Spark UI:", config.SPARK_UI)
import os, json, time
import ipywidgets as w
from poc.loadgen import LoadGen
from poc.obs_admin import MgmtApi, Throttler
from poc import throttle_eval as te

PHASE_S = int(os.environ.get("THROTTLE_PHASE_S", "60"))
DUR = {"baseline": PHASE_S, "contention": PHASE_S, "simulation": PHASE_S,
       "enforced": 2 * PHASE_S, "recovery": PHASE_S}

lg = LoadGen(max_keys=20000)
iso_rows, iso_ok = lg.verify_tenants()
viz.table(pd.DataFrame(iso_rows))
api = MgmtApi(); thr = Throttler(api)
pol = te.effective_policy(thr)
viz.table(pd.DataFrame([{"mode": pol["mode"], "response_code": pol["response_code"],
                         "existing_rules": len(pol["rules"])}]))
viz.banner(iso_ok, "tenant isolation",
           "distinct principals; effective policy read (unchanged)")

**Presenter notes** (hidden)

Shared namespace bucket, separate principals — that is what principal-scoped rules discriminate on.

## Step 2 — Lab rule + schema verification (TC2.2)
The proposed rule: noisy-tenant PUTs capped at 50 TPS, response 429. `verify_rule` checks the schema against the live API before anything is applied.

In [ ]:
NOISY_RULE = {"id":"demo-noisy-put","rateUnit":"TPS","isDisabled":False,
  "filters":{"principals":["urn:ecs:iam::poc:user/tenant-noisy"],
             "actions":["s3:PutObject"]},
  "limit":{"value":50},"scoping":{"per":["principal"]}}
chk = te.verify_rule(thr, NOISY_RULE)
viz.table(pd.DataFrame([{"schema_ok": chk["ok"],
                         "problems": "; ".join(chk["problems"]) or "—"}]))
display(HTML("<pre>"+json.dumps(NOISY_RULE,indent=2)+"</pre>"))
res = results.write("TC2.2", "rule verified", chk["ok"],
                    {"rule": NOISY_RULE["id"], "problems": chk["problems"]})
viz.banner(res["passed"], "TC2.2", "rule schema verified vs live API")

**Presenter notes** (hidden)

principals accept user:<name>, iam::user:<id>, or the full IAM URN; limit.value is transactions per rateUnit.

## Step 3 — Phase 1: baseline (TC2.1)
Critical GETs alone for DUR[baseline] s — reference TPS, P50/P95/P99.

In [ ]:
out = w.Output(); display(out)
seed = lg.start(noisy=False, critical=True)
lg.start_phase("baseline")
chart = viz.LiveChart(lg); chart.mark("baseline")
chart.refresh(out, seconds=DUR["baseline"])
bp = lg.phase_stats()["baseline"]["tenant-critical GET"]
res = results.write("TC2.1", "baseline critical GETs", bp["ok"] > 0,
    {k: round(bp[k],2) for k in ("n","tps","p50","p95","p99","errors")})
viz.banner(res["passed"], "TC2.1 baseline",
           f"{bp['tps']:.1f} TPS, p95 {bp['p95']:.0f} ms, p99 {bp['p99']:.0f} ms")

**Presenter notes** (hidden)

Critical workload alone — rules exist but stay disabled.

## Step 4 — Phase 2: contention, no rules (TC2.3)
Noisy PUTs added on top of critical GETs, **no throttling configured** — the pure offered-load impact this demo must fix.

In [ ]:
lg.add_noisy(threads=6); lg.start_phase("contention"); chart.mark("contention")
chart.refresh(out, seconds=DUR["contention"])
ps_all = lg.phase_stats()
b, c = ps_all["baseline"], ps_all["contention"]
bl, ct = b["tenant-critical GET"], c["tenant-critical GET"]
res = results.write("TC2.3", "contention", ct["ok"] > 0,
    {"baseline_p95": round(bl["p95"],1), "contention_p95": round(ct["p95"],1),
     "contention_p99": round(ct["p99"],1),
     "offered_noisy_tps": round(c.get("tenant-noisy PUT",{}).get("tps",0),1),
     "noisy_ok_tps": round(c.get("tenant-noisy PUT",{}).get("ok",0)/
                           max(1e-9,c["_window"]["seconds"]),1)})
viz.banner(res["passed"], "TC2.3 contention",
           f"critical p95 {bl['p95']:.0f} -> {ct['p95']:.0f} ms")

**Presenter notes** (hidden)

Unrestricted noisy PUTs degrade critical latency — the problem statement. No rules active yet.

## Step 5 — Phase 3: simulation mode, dry-run (TC2.10)
Rule pushed with mode **dry-run** (ObjectScale Simulation Mode). Both workloads continue — the client must see **zero** rejections while the server only counts matches.

In [ ]:
thr.put_config([NOISY_RULE]); thr.set_mode("dryrun")
lg.start_phase("simulation"); chart.mark("simulation (dry-run)")
chart.refresh(out, seconds=DUR["simulation"])
sp = lg.phase_stats()["simulation"]
sn = sp.get("tenant-noisy PUT", {})
srv_sim = te.server_counters(thr)
sim_clean = sn.get("throttled", 0) == 0 and sn.get("errors", 0) == 0
res = results.write("TC2.10", "simulation mode", sim_clean,
    {"client_429_or_slowdown": sn.get("throttled",0),
     "client_errors": sn.get("errors",0),
     "noisy_ok_tps": round(sn.get("ok",0)/max(1e-9,sp["_window"]["seconds"]),1),
     "mode": thr.get_mode(),
     "server_metrics_available": srv_sim["available"]})
viz.banner(res["passed"], "TC2.10 simulation",
           "0 client rejections — dry-run matches are server-side only"
           if sim_clean else "client saw rejections in dry-run!")

**Presenter notes** (hidden)

Dry-run validates rule matching safely: client sees only 200s; simulated-match counters are a server-side concept — probed and reported, never fabricated.

## Step 5 — Phase 3: enforcement, HTTP 429 (TC2.4)
Mode enabled with 429 responses; both workloads continue for DUR[enforced] s. Mid-phase check confirms noisy load stays sustained.

In [ ]:
thr.set_mode("enabled"); thr.set_response_code(429)
time.sleep(2)  # allow config write to settle before readback
eff = {"mode": thr.get_mode(), "response_code": thr.get_response_code(),
       "rules": len(thr.get_rules())}
assert eff["mode"] == "enabled" and eff["response_code"] == 429, eff
lg.start_phase("enforced"); chart.mark("enforce 429")
half = DUR["enforced"] // 2
chart.refresh(out, seconds=half)
sustained_mid = lg.sustained(10, 0.5)
if lg.running("tenant-noisy") and not sustained_mid:
    viz.banner(False, "sustained check", "noisy load dropped — extending watch")
chart.refresh(out, seconds=DUR["enforced"] - half)
sustained_end = lg.sustained(10, 0.5)
ps_all = lg.phase_stats()
ep, cp = ps_all["enforced"], ps_all["contention"]
en, ec = ep.get("tenant-noisy PUT", {}), ep.get("tenant-critical GET", {})
cn = cp.get("tenant-noisy PUT", {})
lim_v = NOISY_RULE["limit"]["value"]
enforced, enf_mode = te.detect(ps_all, lim_v)
data_ok = en.get("n", 0) >= 50 and ec.get("n", 0) >= 50
res = results.write("TC2.4", "enforcement", data_ok,
    {"noisy": {k: round(en.get(k,0),2) for k in
               ("n","tps","ok","throttled","r429","r503_slowdown","r503_other",
                "errors","retries_ok","retries_fail","boundary",
                "p50","p95","p99")},
     "noisy_ok_tps_before": round(cn.get("ok",0)/max(1e-9,cp.get("_window",{}).get("seconds",1)),1),
     "noisy_ok_tps_enforced": round(en.get("ok",0)/max(1e-9,ep.get("_window",{}).get("seconds",1)),1),
     "codes": en.get("codes", {}),
     "critical_p95": round(ec.get("p95",0),1),
     "critical_p99": round(ec.get("p99",0),1),
     "sustained_mid": sustained_mid, "sustained_end": sustained_end,
     "effective_config": eff,
     "enforcement_active": enforced, "enforcement_mode": enf_mode})
viz.banner(enforced, "TC2.4",
           f"enforcement active — {enf_mode} mode" if enforced
           else "no enforcement signature (rejects or delay) — rule may "
                "still be propagating (≤5 min cache)")

**Presenter notes** (hidden)

This build throttles by request delay: all 200s, ok_tps pinned at the limit, noisy latency spikes — verdicts detect both modes.

## Step 7 — Phase 5: continued recovery observation (TC2.5)
**No configuration change here** — enforcement from Step 6 stays active; this phase simply keeps measuring. Noisy PUTs continue at the same offered rate, the rule keeps capping them, and we observe whether critical latency holds at recovery level. Throttle events on the noisy side are expected to stay elevated.

In [ ]:
lg.start_phase("recovery"); chart.mark("recovery (enforcement continues)")
chart.refresh(out, seconds=DUR["recovery"])
rp = lg.phase_stats()["recovery"]
rc, rn = rp.get("tenant-critical GET", {}), rp.get("tenant-noisy PUT", {})
rec_span = max(1e-9, rp["_window"]["seconds"])
res = results.write("TC2.5", "recovery observation under sustained throttling",
    rc.get("ok",0) > 0 and rn.get("n",0) > 0,
    {"critical_p95": round(rc.get("p95",0),1),
     "critical_p99": round(rc.get("p99",0),1),
     "critical_errors": rc.get("errors",0),
     "critical_throttled": rc.get("throttled",0),
     "noisy_offered_tps": round(rn.get("n",0)/rec_span,1),
     "noisy_throttled": rn.get("throttled",0),
     "noisy_ok_tps": round(rn.get("ok",0)/rec_span,1)})
viz.banner(res["passed"], "TC2.5 recovery",
           f"critical p95 {rc.get('p95',0):.0f} ms; noisy still offering "
           f"{rn.get('n',0)/rec_span:.0f} TPS")

**Presenter notes** (hidden)

The mechanism is already-running enforcement, observed longer — not a second action. Elevated noisy throttle events are expected.

## Step 8 — Restart safety check
Stop and restart the noisy group once to prove it is safely restartable, then leave it stopped for the remaining steps.

In [ ]:
leftover = lg.stop_noisy()
restarted = lg.add_noisy(1) == 1
time.sleep(3)
restarted = restarted and lg.running("tenant-noisy")
leftover += lg.stop_noisy()
res = results.write("TC2.11", "worker lifecycle",
                    restarted and leftover == 0,
                    {"noisy_restartable": restarted,
                     "lingering_workers": leftover})
viz.banner(res["passed"], "worker lifecycle",
           "restart ok, no lingering workers" if res["passed"]
           else "worker lifecycle problem")

## Step 9 — Verdicts + server-side correlation (TC2.9)
Score all phases against the approved thresholds; correlate client-side 429s with ObjectScale server counters where the build exposes them.

In [ ]:
verdicts = te.evaluate(lg.phase_stats(), rule=NOISY_RULE)
srv = te.server_counters(thr)
rows = [{"check": v["name"], "verdict": v["verdict"], "detail": v["detail"]}
        for v in verdicts["verdicts"]]
viz.table(pd.DataFrame(rows))
corr = {"client_throttled": sum(v.get("throttled",0) for v in
            lg.phase_stats().get("enforced",{}).values() if isinstance(v,dict)),
        "server_metrics_available": srv["available"], "probed": srv["probed"]}
if srv["available"]:
    corr["server_data"] = srv["data"]
else:
    print("server-side throttle metrics not exposed on this build — "
          "client-side observations only")
res = results.write("TC2.9", "verdicts + correlation",
                    verdicts["overall"] != "FAIL",
                    {"overall": verdicts["overall"], "verdicts": rows, **corr})
viz.banner(verdicts["overall"] != "FAIL", "overall verdict",
           verdicts["overall"])

**Presenter notes** (hidden)

A 50 TPS limit does not yield exactly 50 ok/s — enforcement windows, bursts and retries shift observed rates; thresholds carry tolerance.

## Step 10 — Phase summary
All four phases side by side: TPS, latencies, throttled, errors, codes.

In [ ]:
rows = [{"phase": ph, "workload": k, "n": v["n"], "tps": round(v["tps"],1),
          "p50": round(v["p50"],0), "p95": round(v["p95"],0),
          "p99": round(v["p99"],0), "throttled": v["throttled"],
          "errors": v["errors"], "retries": v["retries"]}
         for ph, gs in lg.phase_stats().items()
         for k, v in gs.items() if k != "_window"]
viz.table(pd.DataFrame(rows))
chart.refresh(out, seconds=2)

**Presenter notes** (hidden)

Shaded bands mark closed phases — the four-phase story in one chart.

## Step 11 — Prefix-scoped rule (TC2.6)
A second rule limits `raw/*` writes only — other prefixes unaffected.

In [ ]:
PRE = dict(NOISY_RULE, id="demo-raw-prefix",
   filters={"principals":["urn:ecs:iam::poc:user/tenant-noisy"],"actions":["s3:PutObject"],
            "paths":{"matcher":"wildcard","values":["poc/poc-throttle/raw/*"]}},
   scoping={"per":["principal","path"]})
thr.put_config([NOISY_RULE, PRE]); chart.mark("prefix rule")
rules = thr.get_rules()
res = results.write("TC2.6", "prefix rule", len(rules) == 2, {"rules": len(rules)})
viz.table(pd.DataFrame(rules)[["id","rateUnit"]]); viz.banner(res["passed"], "TC2.6")

**Presenter notes** (hidden)

paths values are <namespace>/<bucket>/<key-glob>.

## Step 12 — Stop load, Spark write under rules (TC2.7)
Stop the generator and time a Spark write as tenant-noisy under the active rules.

In [ ]:
leftover = lg.stop()
import time as T
t0 = T.time()
spark.range(100_000).write.mode("overwrite").json(
    f"s3a://{config.BUCKET_THROTTLE}/{results.run_id()}/throttled-write")
res = results.write("TC2.7", "spark write under rules", leftover == 0,
                    {"seconds": round(T.time() - t0, 1),
                     "lingering_workers": leftover})
viz.banner(res["passed"], "TC2.7", f"{res['metrics']['seconds']}s")

**Presenter notes** (hidden)

With enforcement active, S3A's throttle retry policy absorbs 429s.

## Step 13 — Restore
Guaranteed restore of the pre-demo throttler backup — registered at exit as well.

In [ ]:
import atexit
atexit.register(thr.restore)
thr.restore()
res = results.write("TC2.8", "restore", thr.get_mode() == "disabled" and not thr.get_rules())
viz.banner(res["passed"], "restore", f"mode={thr.get_mode()}, rules={len(thr.get_rules())}")
spark.stop()

**Presenter notes** (hidden)

Backup: reports/throttle_backup.json — written by scripts/prep_obs.py.